In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os, pathlib, time

# 1) Use a persistent user-data-dir so your login survives across runs
profile_dir = pathlib.Path("./chrome_profile")
profile_dir.mkdir(exist_ok=True)

opts = Options()
opts.add_argument(f"--user-data-dir={profile_dir.resolve()}")
opts.add_argument("--disable-blink-features=AutomationControlled")
opts.add_argument("--start-maximized")
# If you run on a server with no GPU, uncomment:
# opts.add_argument("--disable-gpu")
# opts.add_argument("--no-sandbox")

# Set a good desktop UA (helps avoid “something went wrong”)
opts.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=opts)

# Mask webdriver = true (lightweight; won’t defeat hard checks but helps)
try:
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
        """
    })
except Exception:
    pass

# Set Accept-Language so X returns expected layout and content
try:
    driver.execute_cdp_cmd("Network.enable", {})
    driver.execute_cdp_cmd("Network.setExtraHTTPHeaders", {
        "headers": {
            "Accept-Language": "en-US,en;q=0.9,th;q=0.8"
        }
    })
except Exception:
    pass

driver.get("https://x.com/login")

print("Log in to X/Twitter in this window. Once you see your home timeline, come back and run the next cell.")


Log in to X/Twitter in this window. Once you see your home timeline, come back and run the next cell.


In [136]:
# === Full X/Twitter scraper with raw text + emoji names + embedded quotes/replies ===

import time, re
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
import pandas as pd
import emoji

# === Your SocialMediaRecord imports/constants ===
from scraper.dataclass import SocialMediaRecord  # adjust import
SCRAPER_NAME = "selenium_x_scraper_v1"
PLATFORM = "twitter"
DEFAULT_PLATFORM_TYPE = "post"
CONTENT_TYPE = "text"

# ---------- helpers ----------
def _num_from_label(s):
    if not s: return None
    s = s.strip().replace(",", "")
    m = re.search(r'([\d]+(?:\.\d+)?)\s*([KkMm])?', s)
    if not m: return None
    val = float(m.group(1))
    suf = (m.group(2) or "").lower()
    if suf == "k": val *= 1_000
    elif suf == "m": val *= 1_000_000
    return int(val)

def _is_error_screen(drv):
    try:
        err = drv.find_elements(By.XPATH, "//*[contains(., 'Something went wrong')]")
        return bool(err)
    except Exception:
        return False

def _ensure_logged_in(drv, timeout=10):
    try:
        WebDriverWait(drv, timeout=timeout).until(
            EC.any_of(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="SideNav_NewTweet_Button"], [data-testid="tweetButtonInline"]')),
                EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/compose/tweet"]')),
                EC.presence_of_element_located((By.CSS_SELECTOR, 'a[aria-label*="Profile"]'))
            )
        )
        return True
    except TimeoutException:
        return False

def _wait_for_tweets(drv, timeout=12):
    try:
        WebDriverWait(drv, timeout=timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'article[role="article"]'))
        )
        return True
    except TimeoutException:
        return False

def _safe_reload_until_ok(drv, target_url, tries=4, settle=2.0):
    for i in range(tries):
        if i == 0:
            drv.get(target_url)
        else:
            drv.refresh()
        time.sleep(1.2)
        if _is_error_screen(drv):
            time.sleep(1.2)
            continue
        if _wait_for_tweets(drv, timeout=10):
            time.sleep(settle)
            return True
    return False

# ---------- emoji conversion ----------
def emojify_text(txt: str) -> str:
    """Convert actual emojis in text to :name: format."""
    return emoji.demojize(txt, delimiters=(":", ":"))

# ---------- HTML parsing ----------
def _parse_soup_article(soup):
    row = {
        "tweet_id": None, "timestamp_iso": None,
        "user_display_name": "", "user_handle": "",
        "text": "", "raw_text": "",
        "like_count": None, "retweet_count": None,
        "reply_count": None, "view_count": None,
        "permalink": "", "key": None
    }

    # user display name & handle
    name_block = soup.select_one('div[data-testid="User-Name"], div[data-testid="User-Names"]')
    if name_block:
        spans = name_block.select("span")
        for sp in spans:
            txt = sp.get_text()
            if not txt:
                continue
            if txt.startswith("@"):
                row["user_handle"] = txt
            elif not row["user_display_name"]:
                row["user_display_name"] = txt
    else:
        link = soup.select_one('a[role="link"][href^="/"]')
        if link:
            aria = link.get("aria-label") or ""
            m = re.search(r'^(.*?)\s*\(@', aria)
            if m:
                row["user_display_name"] = m.group(1)

    # RAW text (preserve emojis)
    tw = soup.select_one('div[data-testid="tweetText"]')
    if tw:
        raw = tw.get_text(separator="", strip=False).replace("\u200b", "")
        row["raw_text"] = raw
        row["text"] = emojify_text(raw)

    # timestamp
    t = soup.select_one("time")
    if t and t.has_attr("datetime"):
        row["timestamp_iso"] = t["datetime"]

    # permalink & tweet id
    a = soup.select_one('a[href*="/status/"]')
    if a and a.has_attr("href"):
        href = a["href"]
        if href.startswith("/"):
            href = urljoin("https://x.com", href)
        row["permalink"] = href
        m = re.search(r"/status/(\d+)", href)
        if m:
            row["tweet_id"] = m.group(1)

    # counts
    def c(selector):
        el = soup.select_one(selector)
        if not el:
            return None
        lab = el.get("aria-label") or el.get_text()
        return _num_from_label(lab)
    row["reply_count"]   = c('[data-testid="reply"]')
    row["retweet_count"] = c('[data-testid="retweet"]')
    row["like_count"]    = c('[data-testid="like"]')

    # views
    for s in soup.select('span[aria-label]'):
        lab = s.get("aria-label") or ""
        if "View" in lab:
            vc = _num_from_label(lab)
            if vc is not None:
                row["view_count"] = vc
                break

    row["key"] = row["tweet_id"] or f"{row['user_handle']}|{row['timestamp_iso']}|{(row['text'] or '')[:50]}"
    return row

def _extract_embedded_from_soup(soup):
    embedded = []
    nested = soup.select('article[role="article"]')
    if len(nested) > 1:
        for sub in nested[1:]:
            sub_soup = BeautifulSoup(str(sub), "html.parser")
            sub_row = _parse_soup_article(sub_soup)
            sub_row["is_embedded"] = True
            embedded.append(sub_row)
    card = soup.select_one('div[data-testid="card.body"], div[aria-label*="Quote"]')
    if card:
        sub_article = card.select_one('article[role="article"]')
        if sub_article:
            sub_soup = BeautifulSoup(str(sub_article), "html.parser")
            sub_row = _parse_soup_article(sub_soup)
            sub_row["is_embedded"] = True
            if sub_row["key"] not in {e["key"] for e in embedded}:
                embedded.append(sub_row)
    return embedded

# ---------- main scraper ----------
def scrape_tweets(driver, target_url, max_scrolls=60, pause_secs=1.2, collect_embedded=False):
    if not _ensure_logged_in(driver, timeout=15):
        print("Not logged in; log in manually in Chrome.")
        return []

    ok = _safe_reload_until_ok(driver, target_url, tries=5, settle=1.5)
    if not ok:
        print("Failed to load page.")
        return []

    results, seen = [], set()
    not_growing, last_seen = 0, 0

    def get_articles(drv):
        return drv.find_elements(By.CSS_SELECTOR, 'article[role="article"]')

    for _ in range(max_scrolls):
        if _is_error_screen(driver):
            if not _safe_reload_until_ok(driver, driver.current_url, tries=3, settle=1.2):
                break

        articles = get_articles(driver)
        for idx in range(len(articles)):
            try:
                art = get_articles(driver)[idx]
                html = driver.execute_script("return arguments[0].outerHTML;", art)
                soup = BeautifulSoup(html, "html.parser")
                row = _parse_soup_article(soup)
            except StaleElementReferenceException:
                time.sleep(0.12)
                continue
            except Exception:
                continue

            if not (row.get("tweet_id") or row.get("text")):
                continue
            if row["key"] in seen:
                continue

            seen.add(row["key"])
            results.append(row)
            print(f"[{row['timestamp_iso']}] {row['user_display_name']} {row['user_handle']} "
                  f"♥{row['like_count']} ↻{row['retweet_count']} 💬{row['reply_count']} 👁️{row['view_count']}\n"
                  f"{row['text']}\n{row['permalink']}\n" + "—"*40)

            if collect_embedded:
                for er in _extract_embedded_from_soup(soup):
                    if not (er.get("tweet_id") or er.get("text")):
                        continue
                    if er["key"] in seen:
                        continue
                    seen.add(er["key"])
                    results.append(er)
                    print(f"  (embedded) {er.get('text')[:120]} ...")

        if len(seen) == last_seen:
            not_growing += 1
        else:
            not_growing = 0
        last_seen = len(seen)
        if not_growing >= 12:
            break

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause_secs)

    print(f"\nCollected {len(results)} unique tweets (collect_embedded={collect_embedded}).")
    return results

# ---------- convert to SocialMediaRecord ----------
def infer_platform_type(tweet_text: str) -> str:
    if (tweet_text or "").strip().startswith("RT @"):
        return "retweet"
    return DEFAULT_PLATFORM_TYPE

def tweet_to_record(t: dict) -> SocialMediaRecord:
    raw = t.get("raw_text") or ""
    return SocialMediaRecord(
        id=t.get("tweet_id") or t.get("key"),
        platform=PLATFORM,
        platform_type=infer_platform_type(raw),
        url=t.get("permalink") or "",
        content_type=CONTENT_TYPE,
        timestamp=t.get("timestamp_iso") or "",
        scraper_module=SCRAPER_NAME,
        text=t.get("text") or "",    # emojified text
        raw_text=raw,                # raw emojis preserved
    )

def tweets_to_dataframe_social(tweets: list[dict]) -> pd.DataFrame:
    records = [tweet_to_record(t) for t in tweets]
    df = pd.DataFrame([r.to_dict() for r in records])
    try:
        df = df[SocialMediaRecord.get_fields()]
    except Exception:
        pass
    return df

# === Example usage ===
TARGET_URL = "https://x.com/crta_me/with_replies"
raw_items = scrape_tweets(driver, TARGET_URL, max_scrolls=150, pause_secs=1.2, collect_embedded=True)

records_df = tweets_to_dataframe_social(raw_items)
print(f"Prepared {len(records_df)} records in SocialMediaRecord format (RAW text + emojis).")
display(records_df.head(3))
display(records_df.tail(1))


[2025-07-29T16:26:08.000Z] Maeriberry @crta_me ♥2 ↻20 💬31 👁️None
ชื่อทวิตอ่านว่า มาเอริเบอร์รี่ ค่า
https://x.com/crta_me/status/1950231375304659195
————————————————————————————————————————
[2025-10-22T15:33:11.000Z] louise @first_17 ♥2782 ↻7515 💬5 👁️None
ก็สมควรโดนด่าอะ พล็อตเหี้ยไรเนี่ย pseudo incest มาก อยากรู้ใครตัดสินใจเลือกอันนี้มา เพราะแบบ มันน่าทำจริงๆ หรอ คนตัดสินใจคือแบบ ใช้ไรคิดอะ สติ ศีลธรรม จรรยาบรรณ อยู่ตรงไหน
https://x.com/first_17/status/1981021019516056054
————————————————————————————————————————
[2025-10-23T17:35:25.000Z] Maeriberry @crta_me ♥0 ↻0 💬0 👁️None
มันเป็นแค่พล็อตไหมคะ เรื่องแต่งอะ จะปสด. ด่าเขาทำไมตกมันเหรอ
https://x.com/crta_me/status/1981414170043330738
————————————————————————————————————————
[2025-10-22T03:48:47.000Z] ติดเกาะ @meawmio ♥5727 ↻11469 💬20 👁️None
กุจะบ้า ผช.ที่ภาพลักษณ์ดีมาตลอดนอกใจแฟนไปคั่วดารา AV
https://x.com/meawmio/status/1980843752420962658
————————————————————————————————————————
[2025-10-23T14:14:37.000Z] Maeriberry @crta_me ♥0 ↻0 💬0 

,id,platform,platform_type,url,content_type,timestamp,scraper_module,text,raw_text,scrape_date
0,1950231375304659195,twitter,post,https://x.com/crta_me/status/1950231375304659195,text,2025-07-29T16:26:08.000Z,selenium_x_scraper_v1,ชื่อทวิตอ่านว่า มาเอริเบอร์รี่ ค่า,ชื่อทวิตอ่านว่า มาเอริเบอร์รี่ ค่า,2025-10-23 17:59:34
1,1981021019516056054,twitter,post,https://x.com/first_17/status/1981021019516056054,text,2025-10-22T15:33:11.000Z,selenium_x_scraper_v1,ก็สมควรโดนด่าอะ พล็อตเหี้ยไรเนี่ย pseudo inces...,ก็สมควรโดนด่าอะ พล็อตเหี้ยไรเนี่ย pseudo inces...,2025-10-23 17:59:34
2,1981414170043330738,twitter,post,https://x.com/crta_me/status/1981414170043330738,text,2025-10-23T17:35:25.000Z,selenium_x_scraper_v1,มันเป็นแค่พล็อตไหมคะ เรื่องแต่งอะ จะปสด. ด่าเข...,มันเป็นแค่พล็อตไหมคะ เรื่องแต่งอะ จะปสด. ด่าเข...,2025-10-23 17:59:34


,id,platform,platform_type,url,content_type,timestamp,scraper_module,text,raw_text,scrape_date
455,1950096712338383089,twitter,post,https://x.com/crta_me/status/1950096712338383089,text,2025-07-29T07:31:01.000Z,selenium_x_scraper_v1,ความชอบเราทำให้ใครเดือนร้อนเหรอค้าา มันเป็นเรื...,ความชอบเราทำให้ใครเดือนร้อนเหรอค้าา มันเป็นเรื...,2025-10-23 17:59:34


In [137]:
import pandas as pd
import os

csv_file = r"D:\Documents\University\Senior Project\gender-bias-detection\services\scraper\src\scraper\blog_based\tweets_data.csv"

# Convert scraped data to DataFrame
df_new = tweets_to_dataframe_social(raw_items)

if os.path.exists(csv_file):
    # Load existing CSV
    df_existing = pd.read_csv(csv_file)

    # Combine with new data
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)

    # Drop duplicates based on unique tweet id or permalink
    df_combined.drop_duplicates(subset=["id", "url"], inplace=True)

    # Save back
    df_combined.to_csv(csv_file, index=False)
else:
    df_new.to_csv(csv_file, index=False)


Clean + Check row count

In [138]:
import pandas as pd

csv_file = r"D:\Documents\University\Senior Project\gender-bias-detection\services\scraper\src\scraper\blog_based\tweets_data.csv"

# Load existing CSV
df = pd.read_csv(csv_file)

# Drop duplicates based on tweet ID or URL
df.drop_duplicates(subset=["id", "url"], inplace=True)

# Optionally reset index
df.reset_index(drop=True, inplace=True)

# Save cleaned CSV
df.to_csv(csv_file, index=False)

print(f"CSV cleaned. Total unique rows: {len(df)}")


CSV cleaned. Total unique rows: 4186
